In [13]:
from config import settings
import json
from datetime import datetime, timezone, timedelta
import requests
from bank_client import jwt_gen
import pprint
import os

os.chdir("..")

In [12]:
API_ORIGIN = "https://api.enablebanking.com"
ASPSP_NAME = "Erste Bank"
ASPSP_COUNTRY = "HU"

with open(settings.sessions_info_path, "r") as f:
    session_info = json.load(f)

base_headers = jwt_gen()

# Using the first available account for the following API calls
account_uid = session_info[0]['accounts'][0]['uid']

# Retrieving account balances
r = requests.get(f"{API_ORIGIN}/accounts/{account_uid}/balances", headers=base_headers)
if r.status_code == 200:
    print("Balances:")
    print(r.json())
else:
    print(f"Error response {r.status_code}:", r.text)


# Retrieving account transactions (since 90 days ago)
query = {
    "date_from": (datetime.now(timezone.utc) - timedelta(days=90)).date().isoformat(),
}
continuation_key = None
while True:
    if continuation_key:
        query["continuation_key"] = continuation_key
    r = requests.get(
        f"{API_ORIGIN}/accounts/{account_uid}/transactions",
        params=query,
        headers=base_headers,
    )
    if r.status_code == 200:
        resp_data = r.json()
        print("Transactions:")
        print(resp_data["transactions"])
        continuation_key = resp_data.get("continuation_key")
        if not continuation_key:
            print("No continuation key. All transactions were fetched")
            break
        print(f"Going to fetch more transactions with continuation key {continuation_key}")
    else:
        print(f"Error response {r.status_code}:", r.text)

print("All done!")

FileNotFoundError: [Errno 2] No such file or directory: './secrets/sessions.json'

In [5]:
settings.BANKS[0]["name"]

'Erste Bank'